# Operations Data Lab

Datos de ejemplo. Colocar `01_operaciones_demo.csv` en la misma carpeta del notebook.


In [ ]:
import csv, pathlib
archivo = pathlib.Path('01_operaciones_demo.csv')
assert archivo.exists(), 'Falta 01_operaciones_demo.csv en la carpeta actual'
with archivo.open(encoding='utf-8', newline='') as f:
    filas = list(csv.DictReader(f))
print('Filas de entrada:', len(filas))
print('Columnas:', list(filas[0]))


## 1. Calidad de datos


In [ ]:
ids = [r['id_operacion'] for r in filas]
print('Duplicados:', len(ids)-len(set(ids)))
print('Tiempos vacíos:', sum(not r['tiempo_min'].strip() for r in filas))
limpias = {}
for r in filas: limpias.setdefault(r['id_operacion'], r)
registros = list(limpias.values())
print('Operaciones únicas:', len(registros))


## 2. Indicadores por centro

Los tiempos vacíos se excluyen del promedio.


In [ ]:
from collections import defaultdict
from statistics import mean
centros = defaultdict(list)
for r in registros:
    centros[r['centro']].append(r)
for centro in sorted(centros):
    grupo = centros[centro]
    tiempos = [float(r['tiempo_min']) for r in grupo if r['tiempo_min'].strip()]
    pendientes = sum(r['estado']=='Pendiente' for r in grupo)
    print(f'{centro:7} | Total: {len(grupo)} | Pendientes: {pendientes} | Promedio: {mean(tiempos):.1f} min')


## 3. Consulta SQL local (SQLite)


In [ ]:
import sqlite3
con = sqlite3.connect(':memory:')
con.execute('CREATE TABLE operaciones (id TEXT PRIMARY KEY, centro TEXT, estado TEXT, tiempo_min REAL)')
con.executemany('INSERT INTO operaciones VALUES (?,?,?,?)', [(r['id_operacion'], r['centro'], r['estado'], float(r['tiempo_min']) if r['tiempo_min'].strip() else None) for r in registros])
consulta = "SELECT centro, COUNT(*), SUM(CASE WHEN estado='Pendiente' THEN 1 ELSE 0 END), ROUND(AVG(tiempo_min),1) FROM operaciones GROUP BY centro ORDER BY centro"
print('Centro | Total | Pendientes | Promedio')
for fila in con.execute(consulta): print(fila)
con.close()


## 4. Visualización


In [ ]:
pares=[]
for centro in sorted(centros):
    tiempos=[float(r['tiempo_min']) for r in centros[centro] if r['tiempo_min'].strip()]
    pares.append((centro,mean(tiempos)))
try:
    import matplotlib.pyplot as plt
    plt.figure(figsize=(7,3))
    plt.bar([x[0] for x in pares],[x[1] for x in pares])
    plt.title('Tiempo promedio por centro — datos ficticios')
    plt.ylabel('Minutos'); plt.tight_layout(); plt.show()
except ImportError:
    for centro,prom in pares:
        print(f'{centro:7} ' + '■'*round(prom/2) + f' {prom:.1f} min')
